**Knee Movement Analysis During Single-Leg Squat**
=

> analyze the movement of a person's knee during a single-leg squat



## 1. Import and Install Library


measure 2D projected knee deviation in degrees



In [ ]:

%pip install -q "mediapipe>=0.10.20,<0.11" opencv-python-headless numpy pandas matplotlib scipy gradio


In [ ]:

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp
import scipy
import gradio as gr

from IPython.display import display, Video
from google.colab import files

print("All libraries imported successfully!")
print("MediaPipe version:", mp.__version__)
print("OpenCV version:", cv2.__version__)
print("Gradio version:", gr.__version__)


##2. Upload and Display the Squat Video

> Upload the video, display it inside Colab, and check frame count, frame rate, and duration.

> OpenCV will be used to read the video and retrieve its properties, including frame rate, total frames, and duration.





In [ ]:

# Upload the squat video
uploaded = files.upload()

# Save the uploaded video's filename
video_path = next(iter(uploaded))

print("Video uploaded successfully!")
print("Filename:", video_path)


In [ ]:

# Display the original video
display(Video(
    video_path,
    embed=True,
    width=700
))


In [ ]:

# Open the video using OpenCV
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("OpenCV could not open the uploaded video.")

# Retrieve video properties
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    cap.release()
    raise ValueError("Invalid video frame rate.")

duration = total_frames / fps

# Close the video
cap.release()

print("Video Information")
print("-------------------------")
print("Frame rate:", round(fps, 2), "FPS")
print("Total frames:", total_frames)
print("Duration:", round(duration, 2), "seconds")
print("Resolution:", width, "x", height)


## 3. Detect Anatomical Landmark
Identifies body landmarks in a video frame. The code focuses on the patient’s anatomical left hip (23), knee (25), and ankle (27), then marks and connects these points on the image.

> 23, 25, 27 are predefined landmark IDs from Google's MediaPipe Pose model. (https://developers.google.com/edge/mediapipe/solutions/vision/pose_landmarker?utm_source=chatgpt.com$0) << this link as reference kaa

In [ ]:

# Download the pretrained MediaPipe Pose Landmarker model

import urllib.request

model_path = "pose_landmarker_lite.task"

model_url = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/1/"
    "pose_landmarker_lite.task"
)

if not os.path.exists(model_path):
    urllib.request.urlretrieve(model_url, model_path)

print("MediaPipe model ready!")


In [ ]:

# Read one frame from the squat video

cap = cv2.VideoCapture(video_path)

# Select a frame at any second, I used 3
frame_time = 3.0

cap.set(cv2.CAP_PROP_POS_MSEC, frame_time * 1000)

success, frame = cap.read()
cap.release()

if not success:
    raise ValueError("Could not read the selected video frame.")

# Convert OpenCV BGR image into RGB
rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 7))
plt.imshow(rgb_frame)
plt.title("Original Video Frame at 3 Seconds")
plt.axis("off")
plt.show()


In [ ]:

# Detect left-leg landmarks using MediaPipe

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1
)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb_frame
)

with PoseLandmarker.create_from_options(options) as detector:
    result = detector.detect(mp_image)

if not result.pose_landmarks:
    raise ValueError(
        "No body landmarks detected. Try another frame."
    )

# Get landmarks for the detected person
landmarks = result.pose_landmarks[0]

# MediaPipe landmark IDs
LEFT_HIP = 23
LEFT_KNEE = 25
LEFT_ANKLE = 27

# Convert normalized coordinates to pixels
height, width = rgb_frame.shape[:2]

def to_pixel(landmark):
    return (
        int(landmark.x * width),
        int(landmark.y * height)
    )

hip = to_pixel(landmarks[LEFT_HIP])
knee = to_pixel(landmarks[LEFT_KNEE])
ankle = to_pixel(landmarks[LEFT_ANKLE])

print("Left hip:", hip)
print("Left knee:", knee)
print("Left ankle:", ankle)


In [ ]:

# Draw the detected left leg

annotated_frame = rgb_frame.copy()

# Connect hip -> knee -> ankle with green lines
cv2.line(
    annotated_frame, hip, knee,
    (0, 255, 0), 4
)

cv2.line(
    annotated_frame, knee, ankle,
    (0, 255, 0), 4
)

# Draw red circles on each landmark
for point in [hip, knee, ankle]:
    cv2.circle(
        annotated_frame,
        point,
        8,
        (255, 0, 0),
        -1
    )

# Display the annotated image
plt.figure(figsize=(8, 8))
plt.imshow(annotated_frame)
plt.title("MediaPipe Detection: Left Hip, Knee and Ankle")
plt.axis("off")
plt.show()


## 4.  Calculate the Basic Knee Angle

> Calculate the angle at the patient's left knee using NumPy, then display the result on the image

> calculated using vector mathematics and the dot product formula. A nearly straight leg has an angle close to 180 degrees


> also calculate the deviation from straight alignment using 180 degrees minus the knee angle.



In [ ]:

# Calculate the basic knee angle

import numpy as np

def calculate_knee_angle(hip, knee, ankle):

    # Convert MediaPipe landmarks into 2D coordinates
    H = np.array([hip.x, hip.y])
    K = np.array([knee.x, knee.y])
    A = np.array([ankle.x, ankle.y])

    # Create vectors from the knee
    vector_hip = H - K
    vector_ankle = A - K

    # Calculate vector lengths
    norm_hip = np.linalg.norm(vector_hip)
    norm_ankle = np.linalg.norm(vector_ankle)

    if norm_hip < 1e-8 or norm_ankle < 1e-8:
        return np.nan

    # Calculate cosine using the dot product
    cosine_angle = np.dot(
        vector_hip, vector_ankle
    ) / (norm_hip * norm_ankle)

    # Keep cosine between -1 and 1
    cosine_angle = np.clip(
        cosine_angle, -1.0, 1.0
    )

    # Convert the angle from radians to degrees
    angle = np.degrees(
        np.arccos(cosine_angle)
    )

    return angle

print("Knee angle function ready!")


In [ ]:

# Calculate the left knee angle for the frame

hip_landmark = landmarks[23]
knee_landmark = landmarks[25]
ankle_landmark = landmarks[27]

knee_angle = calculate_knee_angle(
    hip_landmark,
    knee_landmark,
    ankle_landmark
)

knee_deviation = 180 - knee_angle

print("Left Knee Angle:",
      round(knee_angle, 2), "degrees")

print("Deviation from Straight:",
      round(knee_deviation, 2), "degrees")


In [ ]:

# Draw the knee angle on the image

angle_frame = rgb_frame.copy()

# Draw the left leg
cv2.line(angle_frame, hip, knee, (0, 255, 0), 4)
cv2.line(angle_frame, knee, ankle, (0, 255, 0), 4)

# Draw the landmarks
for point in [hip, knee, ankle]:
    cv2.circle(
        angle_frame,
        point,
        8,
        (255, 0, 0),
        -1
    )

# Add angle measurement beside the knee
cv2.putText(
    angle_frame,
    f"{knee_angle:.2f} deg",
    (knee[0] + 15, knee[1] - 15),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (255, 255, 0),
    2
)

# Display the result
plt.figure(figsize=(8, 8))
plt.imshow(angle_frame)
plt.title("Basic Left Knee Angle Measurement")
plt.axis("off")
plt.show()


## 5. Measure Knee Deviation Across the Video

> Create a graph showing how the patient's left knee deviation changes over time





In [ ]:

# Measure knee deviation across the video

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

# Open the uploaded video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

if not cap.isOpened() or fps <= 0:
    cap.release()
    raise ValueError("Cannot open video or invalid FPS.")

# Lists to store results
times = []
knee_angles = []
knee_deviations = []

frame_number = 0

# Configure MediaPipe for video processing
options_video = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1
)

# Create the MediaPipe detector
with mp.tasks.vision.PoseLandmarker.create_from_options(
    options_video
) as detector:

    while True:

        # Read one video frame
        success, frame = cap.read()

        if not success:
            break

        # Convert BGR to RGB
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Convert to MediaPipe image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        # Calculate frame timestamp
        timestamp_ms = int(
            frame_number / fps * 1000
        )

        # Detect landmarks in this frame
        result = detector.detect_for_video(
            mp_image,
            timestamp_ms
        )

        if result.pose_landmarks:

            landmarks = result.pose_landmarks[0]

            # Left hip, knee and ankle
            hip = landmarks[23]
            knee = landmarks[25]
            ankle = landmarks[27]

            # Use the function from Step 4
            angle = calculate_knee_angle(
                hip, knee, ankle
            )

            # Calculate deviation from straight
            deviation = 180 - angle

        else:
            # Missing detection
            angle = np.nan
            deviation = np.nan

        # Save measurements
        times.append(frame_number / fps)
        knee_angles.append(angle)
        knee_deviations.append(deviation)

        frame_number += 1

cap.release()

# Organize results in a DataFrame
df_knee = pd.DataFrame({
    "Time": times,
    "Knee_Angle": knee_angles,
    "Knee_Deviation": knee_deviations
})

print("Video processing completed!")
print("Total frames processed:", frame_number)
print("Valid measurements:",
      df_knee["Knee_Deviation"].notna().sum())

display(df_knee.head())


In [ ]:

# Plot knee deviation over time

plt.figure(figsize=(12, 5))

plt.plot(
    df_knee["Time"],
    df_knee["Knee_Deviation"],
    color="blue",
    linewidth=2,
    label="Left Knee Deviation"
)

# Reference line for straight alignment
plt.axhline(
    y=0,
    color="black",
    linestyle="--",
    label="Straight Alignment"
)

plt.title("Basic Left Knee Deviation During Single-Leg Squat")
plt.xlabel("Time (seconds)")
plt.ylabel("Deviation from Straight (degrees)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

# Summarize knee deviation results

valid = df_knee["Knee_Deviation"].dropna()

if valid.empty:
    print("No valid knee measurements detected.")
else:
    max_idx = df_knee["Knee_Deviation"].idxmax()

    print("Maximum deviation:",
          round(valid.max(), 2), "degrees")

    print("Minimum deviation:",
          round(valid.min(), 2), "degrees")

    print("Time of maximum deviation:",
          round(df_knee.loc[max_idx, "Time"], 2),
          "seconds")


## 6. Improve the Angle Calculation (Signed FPPA)

convert landmarks into pixel coordinates and calculate an oriented 2D knee angle using the dot product and cross product.



In [ ]:

# Step 6: Improved signed 2D knee-angle calculation

import numpy as np

def calculate_fppa(hip, knee, ankle, width, height):

    # Convert normalized landmarks into pixel coordinates
    H = np.array([
        hip.x * width,
        hip.y * height
    ], dtype=float)

    K = np.array([
        knee.x * width,
        knee.y * height
    ], dtype=float)

    A = np.array([
        ankle.x * width,
        ankle.y * height
    ], dtype=float)

    # Create vectors from the knee
    vector_hip = H - K
    vector_ankle = A - K

    # Check that vectors have valid lengths
    if (
        np.linalg.norm(vector_hip) < 1e-8
        or np.linalg.norm(vector_ankle) < 1e-8
    ):
        return np.nan

    # Calculate dot product
    dot_product = np.dot(
        vector_hip,
        vector_ankle
    )

    # Calculate 2D cross product
    cross_product = (
        vector_hip[0] * vector_ankle[1]
        - vector_hip[1] * vector_ankle[0]
    )

    # Calculate oriented angle in degrees
    signed_angle = np.degrees(
        np.arctan2(
            cross_product,
            dot_product
        )
    )

    return signed_angle

print("Improved signed FPPA function ready!")


In [ ]:

# Test the improved calculation on one frame

height, width = rgb_frame.shape[:2]

signed_fppa = calculate_fppa(
    landmarks[23],
    landmarks[25],
    landmarks[27],
    width,
    height
)

print(
    "Basic Knee Angle:",
    round(knee_angle, 2),
    "degrees"
)

print(
    "Improved Signed Angle:",
    round(signed_fppa, 2),
    "degrees"
)

if np.isfinite(signed_fppa):

    # Smallest angular difference from a straight leg
    deviation = 180 - abs(signed_fppa)

    print(
        "Projected Deviation Magnitude:",
        round(deviation, 2),
        "degrees"
    )



## 7. Recalculate Angles for All Frames

improved signed knee-angle function from Section 6 to every frame of the single-leg squat video.

MediaPipe detects the left hip, knee, and ankle in each frame. The calculated angles are stored with their corresponding timestamps using Pandas.

The results are then plotted to visualize how the signed projected knee angle changes throughout the movement.


In [ ]:

# Recalculate signed knee angles for every video frame

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Open the video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

if not cap.isOpened() or fps <= 0:
    cap.release()
    raise ValueError("Cannot open video or invalid FPS.")

# Prepare lists for the results
time_values = []
fppa_values = []

frame_number = 0

# Configure MediaPipe in VIDEO mode
options_fppa = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1
)

# Process the video frame by frame
with mp.tasks.vision.PoseLandmarker.create_from_options(
    options_fppa
) as detector:

    while True:

        success, frame = cap.read()

        if not success:
            break

        # Get actual frame dimensions
        height, width = frame.shape[:2]

        # Convert OpenCV image to RGB
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        # Calculate timestamp in milliseconds
        timestamp_ms = int(
            frame_number / fps * 1000
        )

        # Detect body landmarks
        result = detector.detect_for_video(
            mp_image,
            timestamp_ms
        )

        if result.pose_landmarks:

            landmarks = result.pose_landmarks[0]

            # Use improved function from Step 6
            signed_angle = calculate_fppa(
                landmarks[23],
                landmarks[25],
                landmarks[27],
                width,
                height
            )

        else:
            signed_angle = np.nan

        # Save angle and time
        fppa_values.append(signed_angle)
        time_values.append(frame_number / fps)

        frame_number += 1

cap.release()

# Organize results into a DataFrame
df_fppa = pd.DataFrame({
    "Time": time_values,
    "FPPA_Raw": fppa_values
})

print("Improved angle analysis completed!")
print("Total frames processed:", frame_number)

print(
    "Valid angle measurements:",
    df_fppa["FPPA_Raw"].notna().sum()
)

display(df_fppa.head())


In [ ]:

# Plot raw signed knee angle over time

plt.figure(figsize=(12, 5))

plt.plot(
    df_fppa["Time"],
    df_fppa["FPPA_Raw"],
    color="blue",
    linewidth=2,
    label="Raw Signed Knee Angle"
)

plt.axhline(
    y=-180,
    color="red",
    linestyle="--",
    label="-180 Degree Reference"
)

plt.axhline(
    y=180,
    color="orange",
    linestyle="--",
    label="+180 Degree Reference"
)

plt.title("Raw Signed Left Knee Angle During Single-Leg Squat")
plt.xlabel("Time (seconds)")
plt.ylabel("Raw Signed Angle (degrees)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

# Summarize raw signed angles

valid_fppa = df_fppa["FPPA_Raw"].dropna()

if valid_fppa.empty:
    print("No valid angles were detected.")
else:
    print(
        "Maximum raw signed angle:",
        round(valid_fppa.max(), 2),
        "degrees"
    )

    print(
        "Minimum raw signed angle:",
        round(valid_fppa.min(), 2),
        "degrees"
    )



## 8. Correct the Angle Reference

convert these values into deviations from straight alignment, where 0 degrees represents a straight projected leg.

The corrected values are plotted against time to visualize changes in knee alignment throughout the squat.



> Deviation = 180 - raw angle




In [ ]:

# Step 8: Correct the angle reference

import numpy as np
import pandas as pd

# Convert raw signed angles into deviation from straight
df_fppa["FPPA_Corrected"] = (
    180 - np.abs(df_fppa["FPPA_Raw"])
)

print("Angle reference corrected successfully!")

# Display the first five measurements
display(
    df_fppa[
        ["Time", "FPPA_Raw", "FPPA_Corrected"]
    ].head()
)


In [ ]:

# Plot corrected projected knee deviation

plt.figure(figsize=(12, 5))

plt.plot(
    df_fppa["Time"],
    df_fppa["FPPA_Corrected"],
    color="blue",
    linewidth=2,
    label="Corrected Knee Deviation"
)

# Straight alignment reference
plt.axhline(
    y=0,
    color="black",
    linestyle="--",
    label="Straight Alignment"
)

plt.title(
    "Corrected Left Knee Deviation During Single-Leg Squat"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Deviation from Straight (degrees)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

# Summarize corrected knee deviation

valid = df_fppa["FPPA_Corrected"].dropna()

if valid.empty:
    print("No valid knee deviation measurements.")
else:
    max_idx = df_fppa["FPPA_Corrected"].idxmax()
    min_idx = df_fppa["FPPA_Corrected"].idxmin()

    print(
        "Maximum deviation:",
        round(valid.max(), 2),
        "degrees"
    )

    print(
        "Minimum deviation:",
        round(valid.min(), 2),
        "degrees"
    )

    print(
        "Time of maximum deviation:",
        round(df_fppa.loc[max_idx, "Time"], 2),
        "seconds"
    )

    print(
        "Time of minimum deviation:",
        round(df_fppa.loc[min_idx, "Time"], 2),
        "seconds"
    )



## 9. Detect the Squat Bottom

This section uses MediaPipe to track the vertical distance between the patient's left hip and left ankle throughout the squat video.



In [ ]:

# Step 9: Track hip-to-ankle vertical distance

from scipy.signal import savgol_filter, find_peaks

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

if not cap.isOpened() or fps <= 0:
    cap.release()
    raise ValueError("Could not open video or invalid FPS.")

hip_ankle_distances = []
squat_times = []
frame_number = 0

options_squat = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options_squat
) as detector:

    while True:
        success, frame = cap.read()

        if not success:
            break

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(
            frame_number / fps * 1000
        )

        result = detector.detect_for_video(
            mp_image, timestamp_ms
        )

        if result.pose_landmarks:
            lm = result.pose_landmarks[0]

            hip = lm[23]
            ankle = lm[27]

            # Normalized vertical distance
            distance = ankle.y - hip.y
        else:
            distance = np.nan

        hip_ankle_distances.append(distance)
        squat_times.append(frame_number / fps)

        frame_number += 1

cap.release()

print("Movement tracking completed!")
print("Frames processed:", frame_number)
print(
    "Valid measurements:",
    np.isfinite(hip_ankle_distances).sum()
)


In [ ]:

# Prepare and smooth the movement signal

distances = np.array(
    hip_ankle_distances,
    dtype=float
)

squat_times = np.array(squat_times)

valid = np.isfinite(distances)

if valid.sum() < 5:
    raise ValueError(
        "Not enough valid landmarks to detect a squat."
    )

# Fill missing values using interpolation
distances_filled = np.interp(
    np.arange(len(distances)),
    np.flatnonzero(valid),
    distances[valid]
)

# Choose an odd window length
window = min(31, len(distances_filled))

if window % 2 == 0:
    window -= 1

# Smooth the signal
smooth_distances = savgol_filter(
    distances_filled,
    window_length=window,
    polyorder=3
)

print("Movement signal smoothed successfully!")


In [ ]:

# Detect candidate squat bottoms

prominence = max(
    0.015,
    np.ptp(smooth_distances) * 0.15
)

bottom_indices, properties = find_peaks(
    -smooth_distances,
    distance=max(1, int(fps * 1.0)),
    prominence=prominence
)

bottom_times = squat_times[bottom_indices]

print("Candidate squat bottoms detected:",
      len(bottom_indices))

print("Detected times (seconds):")
print(np.round(bottom_times, 2))

# Select the deepest detected squat bottom
if len(bottom_indices) > 0:

    deepest_index = bottom_indices[
        np.argmin(smooth_distances[bottom_indices])
    ]

    bottom_time = float(
        squat_times[deepest_index]
    )

    print(
        "Deepest squat bottom:",
        round(bottom_time, 2),
        "seconds"
    )

else:
    bottom_time = None

    print(
        "No clear squat bottom detected."
    )


In [ ]:

# Plot squat movement and detected bottoms

plt.figure(figsize=(12, 5))

plt.plot(
    squat_times,
    distances_filled,
    color="lightgray",
    label="Raw Hip-Ankle Distance"
)

plt.plot(
    squat_times,
    smooth_distances,
    color="blue",
    linewidth=2,
    label="Smoothed Hip-Ankle Distance"
)

# Mark all detected squat bottoms
if len(bottom_indices) > 0:
    plt.scatter(
        squat_times[bottom_indices],
        smooth_distances[bottom_indices],
        color="red",
        s=80,
        zorder=5,
        label="Detected Squat Bottom"
    )

plt.title("Squat Bottom Detection")
plt.xlabel("Time (seconds)")
plt.ylabel("Normalized Vertical Hip-Ankle Distance")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



## 10. Inspect Important Video Frames

1. The beginning of the squat.
2. The detected lowest point of the squat.
3. The moment of maximum projected knee deviation.


In [ ]:

# Step 10: Select important timestamps

# 1. Beginning of the squat
start_time = 0.5

# 2. Lowest point from Step 9
# If no squat bottom was detected, use the
# global minimum of the smoothed movement signal.
if bottom_time is None:
    bottom_time = float(
        squat_times[np.argmin(smooth_distances)]
    )
    print("Using global minimum as estimated squat bottom.")

# 3. Maximum projected deviation from Step 8
max_idx = df_fppa["FPPA_Corrected"].idxmax()

max_deviation_time = float(
    df_fppa.loc[max_idx, "Time"]
)

inspection_times = [
    start_time,
    bottom_time,
    max_deviation_time
]

inspection_titles = [
    "Start of Squat",
    "Squat Bottom",
    "Maximum Projected Deviation"
]

print("Selected timestamps:")
for title, time in zip(
    inspection_titles, inspection_times
):
    print(f"{title}: {time:.2f} seconds")


In [ ]:

# Extract and annotate three important video frames

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Could not open the squat video.")

fig, axes = plt.subplots(
    1, 3, figsize=(16, 6)
)

options_inspect = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_poses=1
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options_inspect
) as detector:

    for ax, time, title in zip(
        axes,
        inspection_times,
        inspection_titles
    ):

        # Jump to the selected timestamp
        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            time * 1000
        )

        success, frame = cap.read()

        if not success:
            ax.set_title(f"{title}: Frame unavailable")
            ax.axis("off")
            continue

        # Convert BGR to RGB
        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        height, width = rgb.shape[:2]

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        # Detect landmarks
        result = detector.detect(mp_image)

        # Display original frame
        ax.imshow(rgb)

        if result.pose_landmarks:

            lm = result.pose_landmarks[0]

            # Left hip, knee, ankle in pixels
            points = np.array([
                [lm[i].x * width, lm[i].y * height]
                for i in [23, 25, 27]
            ])

            hip_point = points[0]
            knee_point = points[1]
            ankle_point = points[2]

            # Green lines: actual leg alignment
            ax.plot(
                points[:, 0],
                points[:, 1],
                color="lime",
                linewidth=3
            )

            # Red dots: detected joints
            ax.scatter(
                points[:, 0],
                points[:, 1],
                color="red",
                s=60
            )

            # Yellow dashed hip-to-ankle reference
            ax.plot(
                [hip_point[0], ankle_point[0]],
                [hip_point[1], ankle_point[1]],
                color="yellow",
                linestyle="--",
                linewidth=2
            )

        else:
            ax.text(
                0.5, 0.5,
                "No pose detected",
                transform=ax.transAxes,
                ha="center",
                va="center",
                color="red",
                fontsize=12
            )

        ax.set_title(
            f"{title}\nTime: {time:.2f}s"
        )

        ax.axis("off")

cap.release()

plt.tight_layout()
plt.show()



## 11. Calculate Medial Knee Deviation in Degrees

- Positive (+): Medial knee deviation
- Negative (-): Lateral knee deviation
- Zero (0)


In [ ]:

# Step 11: Calculate signed medial/lateral knee angle

import numpy as np

def calculate_medial_knee_angle(
    left_hip, left_knee, left_ankle,
    right_hip, width, height
):

    # Convert normalized MediaPipe coordinates to pixels
    def to_xy(point):
        return np.array([
            point.x * width,
            point.y * height
        ], dtype=float)

    H = to_xy(left_hip)
    K = to_xy(left_knee)
    A = to_xy(left_ankle)
    RH = to_xy(right_hip)

    # 1. Calculate the projected knee-angle magnitude
    vector_hip = H - K
    vector_ankle = A - K

    norm_hip = np.linalg.norm(vector_hip)
    norm_ankle = np.linalg.norm(vector_ankle)

    if norm_hip < 1e-8 or norm_ankle < 1e-8:
        return np.nan

    cosine = np.clip(
        np.dot(vector_hip, vector_ankle)
        / (norm_hip * norm_ankle),
        -1.0, 1.0
    )

    knee_angle = np.degrees(np.arccos(cosine))
    deviation = 180.0 - knee_angle

    # 2. Determine the medial direction in the image
    # The patient's right hip is toward the body's midline
    medial_direction = np.sign(RH[0] - H[0])

    if medial_direction == 0:
        return np.nan

    # 3. Find the hip-ankle reference X at knee height
    vertical_span = A[1] - H[1]

    if abs(vertical_span) < 1e-8:
        return np.nan

    fraction = (K[1] - H[1]) / vertical_span

    reference_x = (
        H[0] + fraction * (A[0] - H[0])
    )

    # 4. Determine the knee's direction from the reference
    horizontal_offset = K[0] - reference_x

    medial_offset = (
        horizontal_offset * medial_direction
    )

    # Small tolerance to reduce numerical jitter
    tolerance_pixels = 1.0

    if abs(horizontal_offset) <= tolerance_pixels:
        return 0.0

    # 5. Combine angular magnitude and direction
    signed_deviation = (
        np.sign(medial_offset) * deviation
    )

    return float(signed_deviation)


print("Medial knee-angle function ready!")


In [ ]:

# Test signed medial knee angle on one frame

height, width = rgb_frame.shape[:2]

test_medial_angle = calculate_medial_knee_angle(
    landmarks[23],   # Left hip
    landmarks[25],   # Left knee
    landmarks[27],   # Left ankle
    landmarks[24],   # Right hip
    width,
    height
)

print(
    "Signed projected knee angle:",
    round(test_medial_angle, 2),
    "degrees"
)

if not np.isfinite(test_medial_angle):
    print("Measurement unavailable.")

elif test_medial_angle > 0:
    print("Direction: Medial (inward)")

elif test_medial_angle < 0:
    print("Direction: Lateral (outward)")

else:
    print("Direction: Approximately aligned")


In [ ]:

# Analyze signed medial/lateral knee angles
# across the entire squat video

import cv2
import mediapipe as mp
import pandas as pd
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

if not cap.isOpened() or fps <= 0:
    cap.release()
    raise ValueError("Could not open video or invalid FPS.")

medial_times = []
signed_angles = []
frame_number = 0

options_medial = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options_medial
) as detector:

    while True:

        success, frame = cap.read()

        if not success:
            break

        height, width = frame.shape[:2]

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(
            frame_number / fps * 1000
        )

        result = detector.detect_for_video(
            mp_image,
            timestamp_ms
        )

        if result.pose_landmarks:

            lm = result.pose_landmarks[0]

            angle = calculate_medial_knee_angle(
                lm[23],
                lm[25],
                lm[27],
                lm[24],
                width,
                height
            )

        else:
            angle = np.nan

        signed_angles.append(angle)
        medial_times.append(frame_number / fps)

        frame_number += 1

cap.release()

df_medial = pd.DataFrame({
    "Time": medial_times,
    "Signed_Angle": signed_angles
})

# Keep medial magnitude separately.
# Preserve missing measurements as NaN.
df_medial["Medial_Degrees"] = np.where(
    df_medial["Signed_Angle"].notna(),
    np.maximum(df_medial["Signed_Angle"], 0),
    np.nan
)

print("Signed knee-angle analysis completed!")
print("Frames processed:", frame_number)
print(
    "Valid measurements:",
    df_medial["Signed_Angle"].notna().sum()
)

display(df_medial.head())


In [ ]:

# Plot signed knee deviation over time

plt.figure(figsize=(12, 5))

plt.plot(
    df_medial["Time"],
    df_medial["Signed_Angle"],
    color="blue",
    linewidth=2,
    label="Signed Knee Deviation"
)

plt.axhline(
    0,
    color="black",
    linestyle="--",
    label="Straight Alignment"
)

plt.fill_between(
    df_medial["Time"],
    df_medial["Signed_Angle"],
    0,
    where=(df_medial["Signed_Angle"] > 0),
    color="red",
    alpha=0.15,
    label="Medial Deviation"
)

plt.fill_between(
    df_medial["Time"],
    df_medial["Signed_Angle"],
    0,
    where=(df_medial["Signed_Angle"] < 0),
    color="blue",
    alpha=0.10,
    label="Lateral Deviation"
)

plt.title(
    "Left Knee Medial (+) and Lateral (-) Deviation"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Signed Projected Knee Angle (degrees)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

# Find maximum medial and lateral projected angles

valid = df_medial["Signed_Angle"].dropna()

if valid.empty:
    print("No valid signed knee-angle measurements.")

else:
    max_medial = max(0.0, float(valid.max()))
    max_lateral = max(0.0, float(-valid.min()))

    print("Maximum medial knee deviation:",
          round(max_medial, 2), "degrees")

    print("Maximum lateral knee deviation:",
          round(max_lateral, 2), "degrees")

    if (df_medial["Signed_Angle"] > 0).any():
        medial_index = df_medial["Signed_Angle"].idxmax()
        medial_time = df_medial.loc[medial_index, "Time"]

        print("Time of maximum medial deviation:",
              round(medial_time, 2), "seconds")
    else:
        print("No medial deviation detected.")

    if (df_medial["Signed_Angle"] < 0).any():
        lateral_index = df_medial["Signed_Angle"].idxmin()
        lateral_time = df_medial.loc[lateral_index, "Time"]

        print("Time of maximum lateral deviation:",
              round(lateral_time, 2), "seconds")
    else:
        print("No lateral deviation detected.")



## 12. Generate an Annotated Knee Movement Video


The patient's left hip, knee, and ankle are detected in each frame. Colored lines and markers visualize the projected leg alignment, while a yellow hip-to-ankle line provides a straight reference.

The signed knee angle is displayed in degrees:
- Red: ~medial deviation
- Blue: ~lateral deviation
- Green: ~straight alignment


In [ ]:

# Step 12: Generate annotated knee-movement video

import cv2
import mediapipe as mp
import numpy as np
import os
import subprocess

# Open original video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if not cap.isOpened() or fps <= 0 or width <= 0 or height <= 0:
    cap.release()
    raise ValueError("Cannot open the video or invalid properties.")

# Output filenames
raw_output = "annotated_knee_raw.mp4"
annotated_video_path = "annotated_knee_video.mp4"

# Create video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    raw_output,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise ValueError("Cannot create the output video.")

# Configure MediaPipe
options_annotated = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1
)

frame_number = 0

try:
    with mp.tasks.vision.PoseLandmarker.create_from_options(
        options_annotated
    ) as detector:

        while True:

            success, frame = cap.read()

            if not success:
                break

            # Convert frame to RGB for MediaPipe
            rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            timestamp_ms = int(
                frame_number / fps * 1000
            )

            # Detect body landmarks
            result = detector.detect_for_video(
                mp_image,
                timestamp_ms
            )

            if result.pose_landmarks:

                lm = result.pose_landmarks[0]

                # Calculate signed angle using Step 11
                angle = calculate_medial_knee_angle(
                    lm[23],
                    lm[25],
                    lm[27],
                    lm[24],
                    width,
                    height
                )

                # Convert left-leg landmarks to pixels
                points = [
                    (
                        int(lm[i].x * width),
                        int(lm[i].y * height)
                    )
                    for i in [23, 25, 27]
                ]

                hip, knee, ankle = points

                # Choose annotation color
                # OpenCV uses BGR color order
                if not np.isfinite(angle):
                    color = (160, 160, 160)
                    direction = "Unavailable"
                    angle_text = "N/A"

                elif angle > 0:
                    color = (0, 0, 255)   # Red
                    direction = "Medial"
                    angle_text = f"{angle:+.2f} deg"

                elif angle < 0:
                    color = (255, 0, 0)   # Blue
                    direction = "Lateral"
                    angle_text = f"{angle:+.2f} deg"

                else:
                    color = (0, 255, 0)   # Green
                    direction = "Aligned"
                    angle_text = f"{angle:.2f} deg"

                # Draw left hip-knee-ankle chain
                cv2.line(
                    frame, hip, knee,
                    color, 4
                )

                cv2.line(
                    frame, knee, ankle,
                    color, 4
                )

                # Draw circles at landmarks
                for point in points:
                    cv2.circle(
                        frame,
                        point,
                        7,
                        color,
                        -1
                    )

                # Draw straight hip-ankle reference
                cv2.line(
                    frame, hip, ankle,
                    (0, 255, 255), 2
                )

                # Create dark background for readable text
                cv2.rectangle(
                    frame,
                    (10, 10),
                    (min(width - 1, 430), 105),
                    (30, 30, 30),
                    -1
                )

                # Display knee angle
                cv2.putText(
                    frame,
                    f"Knee angle: {angle_text}",
                    (20, 48),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.75,
                    color,
                    2
                )

                # Display direction
                cv2.putText(
                    frame,
                    f"Direction: {direction}",
                    (20, 83),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.75,
                    color,
                    2
                )

            else:
                # Text when no person is detected
                cv2.putText(
                    frame,
                    "Pose not detected",
                    (20, 45),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )

            # Save frame
            writer.write(frame)
            frame_number += 1

finally:
    cap.release()
    writer.release()

# Convert video to H.264 for browser playback
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-i", raw_output,
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-an",
        annotated_video_path
    ],
    check=True
)

print("Annotated video created successfully!")
print("Output file:", annotated_video_path)
print("Frames processed:", frame_number)
print("File size:",
      round(os.path.getsize(annotated_video_path) / 1e6, 2),
      "MB")


In [ ]:

# Verify annotated video output

print("File exists:",
      os.path.exists(annotated_video_path))

print("Output path:", annotated_video_path)



## 13. Display the Annotated Video

This section displays the annotated single-leg squat video generated in Section 12.

The video shows the detected left hip, knee, and ankle, a straight hip-to-ankle reference line, and the estimated medial or lateral knee deviation in degrees.

The annotated video allows us to visually inspect the landmark tracking and verify that the measurements correspond to the patient's movements.


In [ ]:

# Step 13: Display the annotated video

from IPython.display import Video, display
import os

# Check whether the output video exists
if not os.path.exists(annotated_video_path):
    raise FileNotFoundError(
        "Annotated video not found. Please run Step 12 first."
    )

# Display the video in Google Colab
display(
    Video(
        annotated_video_path,
        embed=True,
        width=700
    )
)



## 14. Install Gradio

Gradio will allow users to upload a single-leg squat video and receive an annotated video and estimated medial/lateral knee deviation measurements in degrees.


In [ ]:

# Step 14: Install Gradio

%pip install -q gradio


In [ ]:

# Import Gradio and check installation

import gradio as gr

print("Gradio installed successfully!")
print("Gradio version:", gr.__version__)



## 15. Build the Interactive Gradio App

In [ ]:

# Step 15: Gradio video-analysis function

import os
import cv2
import numpy as np
import mediapipe as mp
import gradio as gr
import subprocess
import tempfile
import uuid

def analyze_squat(video):

    if video is None:
        return None, "Please upload a squat video."

    # Gradio Video passes the uploaded filepath
    video_file = video

    # Check dependencies from earlier sections
    if not os.path.exists(model_path):
        return None, "MediaPipe model not found. Run Step 3."

    # Open the uploaded video
    cap = cv2.VideoCapture(video_file)

    if not cap.isOpened():
        return None, "Cannot open the uploaded video."

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps <= 0 or width <= 0 or height <= 0:
        cap.release()
        return None, "Invalid video properties."

    # Unique output paths so uploads do not overwrite
    # one another
    run_id = uuid.uuid4().hex[:12]
    output_dir = tempfile.mkdtemp(
        prefix="knee_analysis_"
    )

    raw_path = os.path.join(
        output_dir,
        f"{run_id}_raw.mp4"
    )

    output_path = os.path.join(
        output_dir,
        f"{run_id}_annotated.mp4"
    )

    writer = cv2.VideoWriter(
        raw_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    if not writer.isOpened():
        cap.release()
        return None, "Cannot create output video."

    # MediaPipe VIDEO mode
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path=model_path
        ),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_poses=1
    )

    angles = []
    times = []
    frame_number = 0

    try:
        with mp.tasks.vision.PoseLandmarker.create_from_options(
            options
        ) as detector:

            while True:
                success, frame = cap.read()

                if not success:
                    break

                rgb = cv2.cvtColor(
                    frame, cv2.COLOR_BGR2RGB
                )

                image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=rgb
                )

                timestamp_ms = int(
                    frame_number / fps * 1000
                )

                result = detector.detect_for_video(
                    image, timestamp_ms
                )

                angle = np.nan

                if result.pose_landmarks:

                    lm = result.pose_landmarks[0]

                    # Same function used in Step 11
                    angle = calculate_medial_knee_angle(
                        lm[23],
                        lm[25],
                        lm[27],
                        lm[24],
                        width,
                        height
                    )

                    # Pixel coordinates
                    points = [
                        (
                            int(lm[i].x * width),
                            int(lm[i].y * height)
                        )
                        for i in [23, 25, 27]
                    ]

                    hip, knee, ankle = points

                    # Choose color and direction
                    if not np.isfinite(angle):
                        color = (160, 160, 160)
                        direction = "Unavailable"
                        angle_text = "N/A"

                    elif angle > 0:
                        color = (0, 0, 255)
                        direction = "Medial"
                        angle_text = f"{angle:+.2f} deg"

                    elif angle < 0:
                        color = (255, 0, 0)
                        direction = "Lateral"
                        angle_text = f"{angle:+.2f} deg"

                    else:
                        color = (0, 255, 0)
                        direction = "Aligned"
                        angle_text = "0.00 deg"

                    # Draw left-leg connections
                    cv2.line(
                        frame, hip, knee,
                        color, 4
                    )
                    cv2.line(
                        frame, knee, ankle,
                        color, 4
                    )

                    # Draw landmarks
                    for point in points:
                        cv2.circle(
                            frame, point,
                            7, color, -1
                        )

                    # Hip-to-ankle reference
                    cv2.line(
                        frame, hip, ankle,
                        (0, 255, 255), 2
                    )

                    # Text overlay
                    cv2.rectangle(
                        frame,
                        (10, 10),
                        (min(width - 1, 430),
                         min(height - 1, 110)),
                        (30, 30, 30),
                        -1
                    )

                    cv2.putText(
                        frame,
                        f"Knee angle: {angle_text}",
                        (20, 48),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        color,
                        2
                    )

                    cv2.putText(
                        frame,
                        f"Direction: {direction}",
                        (20, 85),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        color,
                        2
                    )

                else:
                    cv2.putText(
                        frame,
                        "Pose not detected",
                        (20, 45),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (0, 0, 255),
                        2
                    )

                # Store results for summary
                angles.append(angle)
                times.append(frame_number / fps)

                writer.write(frame)
                frame_number += 1

    finally:
        cap.release()
        writer.release()

    if frame_number == 0:
        return None, "No video frames were processed."

    # Convert to browser-compatible H.264 MP4
    try:
        subprocess.run(
            [
                "ffmpeg", "-y",
                "-loglevel", "error",
                "-i", raw_path,
                "-c:v", "libx264",
                "-pix_fmt", "yuv420p",
                "-an",
                output_path
            ],
            check=True
        )

    except (subprocess.CalledProcessError,
            FileNotFoundError):
        return None, "Video conversion failed."

    # Calculate summary statistics
    angles = np.array(angles, dtype=float)
    times = np.array(times, dtype=float)

    valid = np.isfinite(angles)

    if not valid.any():
        summary = (
            "No valid knee measurements detected.\n"
            "Please use a front-facing video "
            "with the left leg clearly visible."
        )

        return output_path, summary

    valid_angles = angles[valid]

    maximum_medial = max(
        0.0,
        float(np.max(valid_angles))
    )

    maximum_lateral = max(
        0.0,
        float(-np.min(valid_angles))
    )

    # Create readable result summary
    summary = (
        "LEFT KNEE ANALYSIS RESULTS\n"
        "--------------------------------\n"
        f"Frames analyzed: {frame_number}\n"
        f"Valid measurements: {int(valid.sum())}\n"
        f"Duration: {frame_number / fps:.2f} seconds\n\n"
        f"Maximum medial deviation: "
        f"{maximum_medial:.2f} degrees\n"
        f"Maximum lateral deviation: "
        f"{maximum_lateral:.2f} degrees\n"
    )

    if np.any(angles > 0):
        medial_index = int(np.nanargmax(angles))

        summary += (
            f"Time of maximum medial deviation: "
            f"{times[medial_index]:.2f} seconds\n"
        )
    else:
        summary += "No positive medial angle detected.\n"

    if np.any(angles < 0):
        lateral_index = int(np.nanargmin(angles))

        summary += (
            f"Time of maximum lateral deviation: "
            f"{times[lateral_index]:.2f} seconds\n"
        )
    else:
        summary += "No negative lateral angle detected.\n"

    summary += (
        "\nPositive (+) = estimated medial alignment\n"
        "Negative (-) = estimated lateral alignment\n\n"
        "Note: Experimental 2D projected measurements, "
        "not a clinical diagnosis."
    )

    return output_path, summary


print("Gradio knee-analysis function ready!")


In [ ]:

# Step 15: Create interactive Gradio interface

demo = gr.Interface(
    fn=analyze_squat,

    inputs=gr.Video(
        label="Upload Video",
        sources=["upload"],
        format="mp4"
    ),

    outputs=[
        gr.Video(
            label="Knee Movement Video",
            format="mp4"
        ),

        gr.Textbox(
            label="Result",
            lines=12
        )
    ],

    title="Knee Movement Analysis",

    description=(
        "Upload Video "

    ),

    submit_btn="Analyze  Video"
)

print(" successfully!")


In [ ]:

# Launch Gradio in Google Colab

demo.close()

demo.launch(
    share=True,
    debug=True
)

